### Config and setup

#### Function refresh

In [1]:
%load_ext autoreload
%autoreload 2
# %reload_ext autoreload

In [2]:
import sys
sys.path.insert(0, '../functions')

#### Library imports

In [108]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

import json
import random
import os
import sys
from pathlib import Path
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import mannwhitneyu
from collections import Counter
import time

import seaborn as sns
import matplotlib.pyplot as plt

from captum.attr import IntegratedGradients

torch.manual_seed(25)
np.random.seed(25)
random.seed(25)

#### Functions import

In [4]:
from models import DANN
from data_loading import prepare_dataloaders, load_subject_data_flexible
from metrics import evaluate

from explainability import (
    DANNClassifierWrapper, load_model_for_xai,
    fold_batchnorm, verify_folding,
    lrp_epsilon_conv2d, lrp_zplus_conv2d,
    lrp_linear, lrp_avgpool,
    compute_lrp, run_lrp_full,
    load_lrp_attributions,
    compute_per_class_accuracy_106,
    compute_deeplift, run_deeplift_full,
    load_deeplift_attributions,
    compute_shap, run_shap_full,
    load_shap_attributions,
    compute_ig, run_ig_full,
    load_ig_attributions,
    run_misclassified_single,
    run_and_save_misclassified
)

from xai_utils import (
    assign_score_groups, get_grouped_order,
    get_score_grouped_order, classify_failure_groups,
    compute_lrp_deeplift_agreement,
    compute_method_agreement,
    compute_within_group_consistency,
    print_consistency_results,
    ch16_list,
    build_binary_relevance_table,
    build_multiclass_relevance_table,
    get_top_bottom_channels
)

from xai_plots import (
    plot_group_mean_attribution,
    plot_stopping_criterion_consistency
)

#### Path

In [5]:
base_path = Path("..")
processed_path = base_path / "Data/processed"
results_path = base_path / "results"
ch16_path = base_path / "Data/processed_16ch"

# existing attribution folders for loading LRP and DeepLIFT
lrp_attr_path = results_path / "lrp_attributions"
binary_attr_path = lrp_attr_path / "binary_cdan_16ch"
multiclass_attr_path = lrp_attr_path / "multiclass_16ch"

deeplift_attr_path = results_path / "deeplift_attributions"
binary_deeplift_path = deeplift_attr_path / "binary_cdan_16ch"
multiclass_deeplift_path = deeplift_attr_path / "multiclass_16ch"

per_class_path = results_path / "per_class_accuracy"

shap_attr_path = results_path / "shap_attributions"
binary_shap_path = shap_attr_path / "binary_cdan_16ch"
multiclass_shap_path = shap_attr_path / "multiclass_16ch"

# new IG folders
ig_attr_path = results_path / "ig_attributions"
binary_ig_zero_path = ig_attr_path / "binary_cdan_16ch_zero"
binary_ig_mean_path = ig_attr_path / "binary_cdan_16ch_mean"
multiclass_ig_zero_path = ig_attr_path / "multiclass_16ch_zero"
multiclass_ig_mean_path = ig_attr_path / "multiclass_16ch_mean"

# only creating zero baseline folders since mean is dropped
binary_ig_zero_path.mkdir(parents=True, exist_ok=True)
multiclass_ig_zero_path.mkdir(parents=True, exist_ok=True)

# misclassified data path
binary_misc_path = results_path / "misclassified_attributions" / "binary_cdan_16ch"
multiclass_misc_path = results_path / "misclassified_attributions" / "multiclass_16ch"

In [6]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

#### 16 channel importing from xai_utils

In [7]:
ch16_list = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
             'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']

#### Valid Subjects

In [8]:
# 106 subjects after removing S088, S092, S100 (incompatible sampling rate)
excluded = ['S088', 'S092', 'S100']

valid_subjects = []
for i in range(1, 110):
    subject_id = f'S{str(i).zfill(3)}'
    subject_file = processed_path / f'{subject_id}_processed.npz'
    if subject_file.exists() and subject_id not in excluded:
        valid_subjects.append(subject_id)

print(f"Valid subjects: {len(valid_subjects)}")
print(f"First five: {valid_subjects[:5]}")
print(f"Last five: {valid_subjects[-5:]}")

Valid subjects: 106
First five: ['S001', 'S002', 'S003', 'S004', 'S005']
Last five: ['S105', 'S106', 'S107', 'S108', 'S109']


#### Setup cell for groupings rebuild

In [9]:
consistency_subjects = ['S007', 'S085', 'S108', 'S107', 'S080',
                         'S097', 'S037', 'S006', 'S039', 'S018']

In [10]:
# load kappas
with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    cdan_results = json.load(f)

all_kappas = {
    subj: cdan_results[subj]['kappa_stopping']['final_kappa']
    for subj in valid_subjects
}

# load per-class accuracy
with open(per_class_path / 'multiclass_16ch_per_class_accuracy.json') as f:
    per_class_data = json.load(f)

all_per_class = {}
for subj in valid_subjects:
    pca = per_class_data[subj]['kappa']['per_class_accuracy']
    all_per_class[subj] = {int(cls): float(acc) for cls, acc in pca.items()}

# rebuild groupings
score_groups, boundaries = assign_score_groups(all_kappas, std_multiplier=0.7)

group_a, group_b, neither = classify_failure_groups(all_per_class, all_kappas)
group_a_subjects = [s['subject'] for s in group_a]
group_b_subjects = [s['subject'] for s in group_b]

category_groups_106 = {}
for subj in valid_subjects:
    if all_kappas[subj] < 0.1:
        category_groups_106[subj] = 'negative_transfer'
    elif subj in group_a_subjects:
        category_groups_106[subj] = 'group_a_collapse'
    elif subj in group_b_subjects:
        category_groups_106[subj] = 'group_b_flat'
    else:
        category_groups_106[subj] = 'neither'

xai_subjects = {
    "top": ["S007", "S085"],
    "bottom_kappa": ["S018", "S097", "S107"],
    "group_a_collapse": ["S037", "S006"],
    "group_b_flat": ["S039", "S020"],
    "typical": ["S080", "S065", "S030", "S108"],
}
xai_subset = [s for group in xai_subjects.values() for s in group]

# load existing attributions for comparison
binary_attrs_106 = load_lrp_attributions(
    binary_attr_path, valid_subjects, n_classes=2)

mc_attrs_106 = load_lrp_attributions(
    multiclass_attr_path, valid_subjects, n_classes=4)

binary_dl_106 = load_deeplift_attributions(
    binary_deeplift_path, valid_subjects, n_classes=2)

mc_dl_106 = load_deeplift_attributions(
    multiclass_deeplift_path, valid_subjects, n_classes=4)

binary_shap_106 = load_shap_attributions(
    binary_shap_path, valid_subjects, n_classes=2)

mc_shap_106 = load_shap_attributions(
    multiclass_shap_path, valid_subjects, n_classes=4)

binary_ig_106 = load_ig_attributions(
    binary_ig_zero_path, valid_subjects, n_classes=2)

mc_ig_106 = load_ig_attributions(
    multiclass_ig_zero_path, valid_subjects, n_classes=4)

print("Setup complete")
print(f"Score groups: {dict(Counter(score_groups.values()))}")
print(f"Category groups: {dict(Counter(category_groups_106.values()))}")
print(f"XAI subset: {len(xai_subset)} subjects")
print(f"LRP binary: {len(binary_attrs_106)}, DeepLIFT: {len(binary_dl_106)}, "
      f"SHAP: {len(binary_shap_106)}, IG: {len(binary_ig_106)}")

Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Setup complete
Score groups: {'mid_high': 21, 'high': 30, 'negative_transfer': 3, 'low': 28, 'mid_low': 24}
Category groups: {'neither': 87, 'negative_transfer': 3, 'group_a_collapse': 13, 'group_b_flat': 3}
XAI subset: 13 subjects
LRP binary: 106, DeepLIFT: 106, SHAP: 106, IG: 106


### Verification cells for thesis writing

#### 3.4.1

In [11]:
from models import EEGNet
import torch
model = EEGNet(n_channels=16, n_timepoints=480)
for name, param in model.named_parameters():
    print(name, param.shape)

temporal_conv.0.weight torch.Size([8, 1, 1, 64])
temporal_conv.1.weight torch.Size([8])
temporal_conv.1.bias torch.Size([8])
depthwise_conv.0.weight torch.Size([16, 1, 16, 1])
depthwise_conv.1.weight torch.Size([16])
depthwise_conv.1.bias torch.Size([16])
separable_conv.0.weight torch.Size([16, 1, 1, 16])
separable_conv.1.weight torch.Size([16, 16, 1, 1])
separable_conv.2.weight torch.Size([16])
separable_conv.2.bias torch.Size([16])
projection.weight torch.Size([128, 240])
projection.bias torch.Size([128])


In [12]:
from models import ConditionalDomainDiscriminator
import inspect
print(inspect.signature(ConditionalDomainDiscriminator.__init__))

(self, embed_dim=128, n_classes=2, dropout_rate=0.5)


In [13]:
from models import EEGNet
import torch
model = EEGNet(n_channels=16, n_timepoints=480)
x = torch.randn(1, 1, 16, 480)
with torch.no_grad():
    for name, layer in model.named_children():
        x_temp = x
        print(f"{name}: {x.shape} -> ", end="")
        try:
            x = layer(x)
            print(x.shape)
        except:
            print("error")

temporal_conv: torch.Size([1, 1, 16, 480]) -> torch.Size([1, 8, 16, 481])
depthwise_conv: torch.Size([1, 8, 16, 481]) -> torch.Size([1, 16, 1, 120])
separable_conv: torch.Size([1, 16, 1, 120]) -> torch.Size([1, 16, 1, 15])
projection: torch.Size([1, 16, 1, 15]) -> error


In [14]:
from models import EEGNet
import torch
model = EEGNet(n_channels=16, n_timepoints=480)
x = torch.randn(1, 1, 16, 480)
with torch.no_grad():
    # run through conv layers manually
    x = model.temporal_conv(x)
    print("after temporal_conv:", x.shape)
    x = model.depthwise_conv(x)
    print("after depthwise_conv:", x.shape)
    x = model.separable_conv(x)
    print("after separable_conv:", x.shape)
    x = x.flatten(1)
    print("after flatten:", x.shape)
    x = model.projection(x)
    print("after projection:", x.shape)

after temporal_conv: torch.Size([1, 8, 16, 481])
after depthwise_conv: torch.Size([1, 16, 1, 120])
after separable_conv: torch.Size([1, 16, 1, 15])
after flatten: torch.Size([1, 240])
after projection: torch.Size([1, 128])


In [15]:
import torch
import torch.nn as nn
conv = nn.Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
x = torch.randn(1, 1, 16, 480)
out = conv(x)
print("input timepoints:", x.shape[3])
print("output timepoints:", out.shape[3])
print("padding:", conv.padding)
print("kernel size:", conv.kernel_size)

input timepoints: 480
output timepoints: 481
padding: (0, 32)
kernel size: (1, 64)


In [16]:
import torch
import torch.nn as nn
conv = nn.Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
print("dilation:", conv.dilation)
print("padding mode:", conv.padding_mode)

dilation: (1, 1)
padding mode: zeros


In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

conv = nn.Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
x = torch.randn(1, 1, 16, 480)

# manually apply padding and check size
x_padded = F.pad(x, (32, 32, 0, 0))
print("after manual padding:", x_padded.shape)

# now apply just the conv without padding
conv_no_pad = nn.Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=0, bias=False)
conv_no_pad.weight = conv.weight
out_manual = conv_no_pad(x_padded)
print("output with manual padding:", out_manual.shape)

out_auto = conv(x)
print("output with auto padding:", out_auto.shape)

after manual padding: torch.Size([1, 1, 16, 544])
output with manual padding: torch.Size([1, 8, 16, 481])
output with auto padding: torch.Size([1, 8, 16, 481])


#### 3.4.2

In [18]:
from models import ConditionalDomainDiscriminator, DANN
import torch

model = DANN(n_channels=16, n_timepoints=480, n_classes=2,
             use_conditional_discriminator=True)
print(model.domain_discriminator)

ConditionalDomainDiscriminator(
  (grl): GradientReversalLayer()
  (network): Sequential(
    (0): Linear(in_features=256, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.5, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)


#### 3.4.4 eegsym

In [19]:
from models import EEGSymAdapted
import inspect
print(inspect.signature(EEGSymAdapted.__init__))

(self, n_channels=8, n_timepoints=480, embed_dim=128, dropout_rate=0.4, n_filters=8)


#### 3.5.1

In [20]:
from training_utils import get_optimizer, device
import inspect
print(inspect.signature(get_optimizer))
print(device)

(model, choice='adamw')
mps


In [21]:
from training_utils import learning_rate, weight_decay, get_optimizer
print("learning_rate:", learning_rate)
print("weight_decay:", weight_decay)

learning_rate: 0.001
weight_decay: 0.01


In [22]:
from pipeline import train_dann_single_fold
import inspect
print(inspect.signature(train_dann_single_fold))

(target_subject, valid_subjects, stopping_criterion='kappa', n_epochs_max=100, batch_size=32, optimizer_choice='adamw', embed_dim=128, dropout_rate=0.25, lambda_domain=0.6, early_stopping_patience=10, early_stopping_min_delta=0.001, composite_threshold=1.0, min_epochs=20, warmup_epochs=10, use_gradient_penalty=True, use_conditional_discriminator=False, use_entropy_weighting=True, use_riemannian_alignment=False, use_adabn=False, feature_extractor_type='eegnet', use_augmentation=False, use_multiclass=False, use_windowed_data=False, n_classes=2, use_16ch=False, window_start_sec=0.5, window_end_sec=2.5, sfreq=160, save_checkpoint=False, checkpoint_tag=None)


In [23]:
from models import EEGNet
import inspect
print(inspect.signature(EEGNet.__init__))

(self, n_channels=8, n_timepoints=480, embed_dim=128, dropout_rate=0.5, f1=8, d=2, f2=16)


In [24]:
from pipeline import train_dann_single_fold
import inspect
sig = inspect.signature(train_dann_single_fold)
print(sig.parameters['dropout_rate'].default)

0.25


In [25]:
from models import DomainDiscriminator, ConditionalDomainDiscriminator
import inspect
print(inspect.signature(DomainDiscriminator.__init__))
print(inspect.signature(ConditionalDomainDiscriminator.__init__))

(self, embed_dim=128, dropout_rate=0.5)
(self, embed_dim=128, n_classes=2, dropout_rate=0.5)


In [26]:
import inspect
from pipeline import train_dann_single_fold
source = inspect.getsource(train_dann_single_fold)
# find the DANN instantiation line
for line in source.split('\n'):
    if 'DANN(' in line or 'dropout' in line.lower():
        print(line)

    dropout_rate=0.25,
    model = DANN(
        dropout_rate=dropout_rate,


#### 3.5.4

In [27]:
from pipeline import train_dann_single_fold
import inspect
sig = inspect.signature(train_dann_single_fold)
print(sig.parameters['composite_threshold'].default)
print(sig.parameters['early_stopping_patience'].default)
print(sig.parameters['early_stopping_min_delta'].default)

1.0
10
0.001


In [28]:
from training_utils import EarlyStopping
import inspect
print(inspect.signature(EarlyStopping.__init__))


(self, patience=10, min_delta=0.001, mode='max')


In [29]:
from pipeline import train_dann_single_fold
import inspect
source = inspect.getsource(train_dann_single_fold)
for i, line in enumerate(source.split('\n')):
    if 'composite_threshold' in line:
        # print surrounding context
        lines = source.split('\n')
        for j in range(max(0,i-2), min(len(lines),i+5)):
            print(lines[j])
        print("---")

    early_stopping_patience=10,
    early_stopping_min_delta=0.001,
    composite_threshold=1.0,
    min_epochs=20,
    warmup_epochs=10,
    use_gradient_penalty=True,
    use_conditional_discriminator=False,
---


In [30]:
from pipeline import train_dann_single_fold
import inspect
source = inspect.getsource(train_dann_single_fold)
# find all occurrences
for i, line in enumerate(source.split('\n')):
    if 'composite_threshold' in line:
        print(f"Line {i}: {line}")

Line 12:     composite_threshold=1.0,


#### 3.5.6

In [31]:
from training_utils import get_optimizer
import inspect
print(inspect.getsource(get_optimizer))


def get_optimizer(model, choice=optimizer_choice):
    """
    returns optimizer with separate learning rates
    discriminator uses higher learning rate to maintain
    adversarial balance against gradient reversal
    """
    discriminator_params = list(model.domain_discriminator.parameters())
    discriminator_param_ids = set(id(p) for p in discriminator_params)
    other_params = [
        p for p in model.parameters() if id(p) not in discriminator_param_ids
    ]

    if choice == "adam":
        return optim.Adam(
            [
                {"params": other_params, "lr": learning_rate},
                {"params": discriminator_params, "lr": learning_rate * 10},
            ]
        )
    elif choice == "adamw":
        return optim.AdamW(
            [
                {
                    "params": other_params,
                    "lr": learning_rate,
                    "weight_decay": weight_decay,
                },
                {
                    "params": discrim

In [32]:
import json
from pathlib import Path
results_path = Path('results')
with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    data = json.load(f)
import numpy as np
kappas_kappa = [data[s]['kappa_stopping']['final_kappa'] for s in data]
kappas_composite = [data[s]['composite_stopping']['final_kappa'] for s in data]
print(f"Kappa stopping mean: {np.mean(kappas_kappa):.4f}")
print(f"Composite stopping mean: {np.mean(kappas_composite):.4f}")
print(f"Kappa stopping above 0.1: {sum(k > 0.1 for k in kappas_kappa)}/106")
print(f"Composite stopping above 0.1: {sum(k > 0.1 for k in kappas_composite)}/106")

Kappa stopping mean: 0.5332
Composite stopping mean: 0.5307
Kappa stopping above 0.1: 103/106
Composite stopping above 0.1: 104/106


In [33]:
import json
import numpy as np
from pathlib import Path

results_path = Path('results')

files = {
    'baseline': 'loso_results_baseline.json',
    'no_warmup_dann': 'loso_results_no_warmup.json',
    'warmup_dann': 'loso_results_warmup.json',
    'gp_warmup_8ch': 'loso_results_gp_warmup.json',
    'cdan_8ch': 'loso_results_cdan.json',
    'cdan_seed42': 'loso_results_cdan_seed42.json',
    'cdan_seed7': 'loso_results_cdan_seed7.json',
    'gp_warmup_16ch': 'loso_results_gp_warmup_16ch.json',
    'cdan_16ch': 'loso_results_cdan_16ch_RETRAIN.json',
    'multiclass_8ch': 'loso_results_multiclass.json',
    'multiclass_16ch': 'loso_results_multiclass_16ch.json',
    'eegsym': 'loso_results_eegsym.json'
}

for name, fname in files.items():
    fpath = results_path / fname
    if fpath.exists():
        with open(fpath) as f:
            data = json.load(f)
        for criterion in ['kappa_stopping', 'composite_stopping']:
            kappas = [data[s][criterion]['final_kappa']
                     for s in data if criterion in data[s]]
            if kappas:
                print(f"{name} {criterion}: mean={np.mean(kappas):.4f}, above_0.1={sum(k>0.1 for k in kappas)}/{len(kappas)}")
    else:
        print(f"{name}: file not found")

no_warmup_dann kappa_stopping: mean=0.4918, above_0.1=102/106
no_warmup_dann composite_stopping: mean=0.5039, above_0.1=101/106
warmup_dann kappa_stopping: mean=0.5099, above_0.1=103/106
warmup_dann composite_stopping: mean=0.5142, above_0.1=103/106
gp_warmup_8ch kappa_stopping: mean=0.5154, above_0.1=105/106
gp_warmup_8ch composite_stopping: mean=0.5070, above_0.1=103/106
cdan_8ch kappa_stopping: mean=0.5180, above_0.1=102/106
cdan_8ch composite_stopping: mean=0.5224, above_0.1=104/106
cdan_seed42 kappa_stopping: mean=0.5160, above_0.1=104/106
cdan_seed42 composite_stopping: mean=0.5278, above_0.1=103/106
cdan_seed7 kappa_stopping: mean=0.5140, above_0.1=104/106
cdan_seed7 composite_stopping: mean=0.5142, above_0.1=104/106
gp_warmup_16ch kappa_stopping: mean=0.5318, above_0.1=103/106
gp_warmup_16ch composite_stopping: mean=0.5359, above_0.1=104/106
cdan_16ch kappa_stopping: mean=0.5332, above_0.1=103/106
cdan_16ch composite_stopping: mean=0.5307, above_0.1=104/106
multiclass_8ch kappa

In [34]:
import json
import numpy as np
from pathlib import Path

results_path = Path('results')
with open(results_path / 'loso_results_baseline.json') as f:
    data = json.load(f)

first_subject = list(data.keys())[0]
print("Keys for first subject:", list(data[first_subject].keys()))

Keys for first subject: ['final_accuracy', 'final_kappa', 'final_auc', 'log']


In [35]:
import json
import numpy as np
from pathlib import Path

results_path = Path('results')
with open(results_path / 'loso_results_baseline.json') as f:
    data = json.load(f)

kappas = [data[s]['final_kappa'] for s in data]
print(f"baseline: mean={np.mean(kappas):.4f}, above_0.1={sum(k>0.1 for k in kappas)}/{len(kappas)}")

baseline: mean=0.4996, above_0.1=102/106


#### 3.6.3


In [36]:
from explainability import compute_deeplift
import inspect
print(inspect.signature(compute_deeplift))

(subject, checkpoint_tag, n_classes=2, use_multiclass=False, stopping_criterion='kappa', baseline='mean', trial_type='correct')


### 3.6.4

In [37]:
from explainability import compute_shap
import inspect
print(inspect.signature(compute_shap))

(subject, checkpoint_tag, valid_subjects, n_classes=2, use_multiclass=False, stopping_criterion='kappa', background_fraction=0.6, trials_per_source_subject=26, trial_type='correct', seed=25)


#### 3.6.5

In [38]:
from explainability import compute_ig
import inspect
print(inspect.signature(compute_ig))

(subject, checkpoint_tag, n_classes=2, use_multiclass=False, stopping_criterion='kappa', baseline='zero', n_steps=50, trial_type='correct')


#### 3.6.6

In [39]:
from explainability import load_shap_attributions
import inspect
print(inspect.signature(load_shap_attributions))

(attr_save_path, valid_subjects, n_classes=2)


In [40]:
import numpy as np
from pathlib import Path

results_path = Path('results')
# valid_subjects = [f'S{str(i).zfill(3)}' for i in range(1, 110)
#                   if f'S{str(i).zfill(3)}' not in ['S088', 'S092', 'S100']]

shap_path = results_path / 'shap_attributions' / 'binary_cdan_16ch'
ig_path = results_path / 'ig_attributions' / 'binary_cdan_16ch_zero'

from explainability import load_shap_attributions, load_ig_attributions
shap_data = load_shap_attributions(shap_path, valid_subjects, n_classes=2)
ig_data = load_ig_attributions(ig_path, valid_subjects, n_classes=2)

print("SHAP keys:", list(shap_data.keys())[:3])
print("IG keys:", list(ig_data.keys())[:3])

Loaded: 106/106 subjects
Loaded: 106/106 subjects
SHAP keys: ['S001', 'S002', 'S003']
IG keys: ['S001', 'S002', 'S003']


In [41]:
# check the structure of one subject
subject = 'S001'
print("SHAP S001 keys:", list(shap_data[subject].keys()))
print("IG S001 keys:", list(ig_data[subject].keys()))

SHAP S001 keys: ['kappa', 'composite']
IG S001 keys: ['kappa', 'composite']


In [42]:
subject = 'S001'
print("SHAP kappa keys:", list(shap_data[subject]['kappa'].keys()))
print("IG kappa keys:", list(ig_data[subject]['kappa'].keys()))

SHAP kappa keys: ['mean_by_class', 'n_trials']
IG kappa keys: ['mean_by_class', 'n_trials']


In [43]:
import numpy as np

correlations_left = []
correlations_right = []

for subject in valid_subjects:
    if subject in shap_data and subject in ig_data:
        shap_left = shap_data[subject]['kappa']['mean_by_class'][0].flatten()
        shap_right = shap_data[subject]['kappa']['mean_by_class'][1].flatten()
        ig_left = ig_data[subject]['kappa']['mean_by_class'][0].flatten()
        ig_right = ig_data[subject]['kappa']['mean_by_class'][1].flatten()

        correlations_left.append(np.corrcoef(shap_left, ig_left)[0,1])
        correlations_right.append(np.corrcoef(shap_right, ig_right)[0,1])

print(f"SHAP vs IG left: r={np.mean(correlations_left):.4f}")
print(f"SHAP vs IG right: r={np.mean(correlations_right):.4f}")

SHAP vs IG left: r=0.3730
SHAP vs IG right: r=0.3567


In [44]:
valid_subjects = [f'S{str(i).zfill(3)}' for i in range(1, 110)
                  if f'S{str(i).zfill(3)}' not in ['S088', 'S092', 'S100']]

# load multiclass attributions
lrp_mc = load_lrp_attributions(multiclass_attr_path, valid_subjects, n_classes=4)
dl_mc = load_deeplift_attributions(multiclass_deeplift_path, valid_subjects, n_classes=4)
shap_mc = load_shap_attributions(multiclass_shap_path, valid_subjects, n_classes=4)
ig_mc = load_ig_attributions(multiclass_ig_zero_path, valid_subjects, n_classes=4)

print("Loaded successfully")
print("LRP keys example:", list(lrp_mc['S001'].keys()))

Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded successfully
LRP keys example: ['kappa', 'composite']


In [45]:
def compute_pairwise_corr(method1, method2, valid_subjects, n_classes=4, criterion='kappa'):
    corrs_per_class = [[] for _ in range(n_classes)]
    for subject in valid_subjects:
        if subject in method1 and subject in method2:
            for c in range(n_classes):
                m1 = method1[subject][criterion]['mean_by_class'][c]
                m2 = method2[subject][criterion]['mean_by_class'][c]
                if m1 is not None and m2 is not None:
                    corrs_per_class[c].append(
                        np.corrcoef(m1.flatten(), m2.flatten())[0,1])
    return [np.mean(corrs) for corrs in corrs_per_class]

pairs = [
    ('LRP', lrp_mc, 'DeepLIFT', dl_mc),
    ('LRP', lrp_mc, 'IG', ig_mc),
    ('LRP', lrp_mc, 'SHAP', shap_mc),
    ('DeepLIFT', dl_mc, 'IG', ig_mc),
    ('DeepLIFT', dl_mc, 'SHAP', shap_mc),
    ('SHAP', shap_mc, 'IG', ig_mc),
]

class_names = ['Left', 'Right', 'Both Fists', 'Both Feet']

for name1, m1, name2, m2 in pairs:
    corrs = compute_pairwise_corr(m1, m2, valid_subjects)
    corr_str = ', '.join([f'{name}={r:.4f}'
                          for name, r in zip(class_names, corrs)])
    print(f"{name1} vs {name2}: {corr_str}")


LRP vs DeepLIFT: Left=0.9246, Right=0.9376, Both Fists=0.9360, Both Feet=0.9348
LRP vs IG: Left=0.9421, Right=0.9536, Both Fists=0.9582, Both Feet=0.9477
LRP vs SHAP: Left=0.4121, Right=0.4422, Both Fists=0.4320, Both Feet=0.5201
DeepLIFT vs IG: Left=0.9788, Right=0.9796, Both Fists=0.9747, Both Feet=0.9825
DeepLIFT vs SHAP: Left=0.3879, Right=0.4234, Both Fists=0.4085, Both Feet=0.5017
SHAP vs IG: Left=0.4051, Right=0.4374, Both Fists=0.4254, Both Feet=0.5163


#### 3.7.2

In [46]:
from metrics import compute_pre_adaptation_mmd
import inspect
print(inspect.signature(compute_pre_adaptation_mmd))

(source_loader, target_loader)


3.7.3

In [47]:
from xai_utils import assign_score_groups, classify_failure_groups
import inspect
print(inspect.signature(assign_score_groups))
print(inspect.signature(classify_failure_groups))

(kappas_dict, std_multiplier=0.7, chance_threshold=0.1)
(all_per_class, all_kappas, spread_threshold=0.4, min_acc_threshold=0.2, flat_spread_threshold=0.2, flat_kappa_threshold=0.3)


#### Binary 16 chann

In [ ]:
results_path = Path('../results')
with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    data = json.load(f)

accuracies = [data[s]['kappa_stopping']['final_accuracy'] for s in data]
aucs = [data[s]['kappa_stopping']['final_auc'] for s in data]

print(f"Mean accuracy kappa stopping: {np.mean(accuracies):.4f}")
print(f"Mean AUC kappa stopping: {np.mean(aucs):.4f}")

accuracies_comp = [data[s]['composite_stopping']['final_accuracy'] for s in data]
aucs_comp = [data[s]['composite_stopping']['final_auc'] for s in data]
print(f"Mean accuracy composite stopping: {np.mean(accuracies_comp):.4f}")
print(f"Mean AUC composite stopping: {np.mean(aucs_comp):.4f}")

Mean accuracy kappa stopping: 0.7661
Mean AUC kappa stopping: 0.7818
Mean accuracy composite stopping: 0.7654
Mean AUC composite stopping: 0.7832


In [72]:
results_path = Path('../results')

with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    bin_data = json.load(f)

bin_k = [bin_data[s]['kappa_stopping']['final_kappa'] for s in bin_data]
bin_c = [bin_data[s]['composite_stopping']['final_kappa'] for s in bin_data]

print("BINARY 16ch kappa stopping:")
print(f"  Mean: {np.mean(bin_k):.4f}")
print(f"  Median: {np.median(bin_k):.4f}")
print(f"  Std: {np.std(bin_k):.4f}")
print(f"  Above 0.1: {sum(k>0.1 for k in bin_k)}/106")

print("\nBINARY 16ch composite stopping:")
print(f"  Mean: {np.mean(bin_c):.4f}")
print(f"  Median: {np.median(bin_c):.4f}")
print(f"  Std: {np.std(bin_c):.4f}")
print(f"  Above 0.1: {sum(k>0.1 for k in bin_c)}/106")


BINARY 16ch kappa stopping:
  Mean: 0.5332
  Median: 0.5133
  Std: 0.2442
  Above 0.1: 103/106

BINARY 16ch composite stopping:
  Mean: 0.5307
  Median: 0.5197
  Std: 0.2360
  Above 0.1: 104/106


### Multiclass 16 

In [50]:
with open(results_path / 'loso_results_multiclass_16ch.json') as f:
    mc_data = json.load(f)

first = list(mc_data.keys())[0]
print("Keys:", list(mc_data[first].keys()))
print("Kappa stopping keys:", list(mc_data[first]['kappa_stopping'].keys()))

Keys: ['kappa_stopping', 'composite_stopping']
Kappa stopping keys: ['final_accuracy', 'final_kappa', 'final_auc', 'log']


In [ ]:
mc_accuracies_k = [mc_data[s]['kappa_stopping']['final_accuracy'] for s in mc_data]
mc_kappas_k = [mc_data[s]['kappa_stopping']['final_kappa'] for s in mc_data]
mc_aucs_k = [mc_data[s]['kappa_stopping']['final_auc'] for s in mc_data]

mc_accuracies_c = [mc_data[s]['composite_stopping']['final_accuracy'] for s in mc_data]
mc_kappas_c = [mc_data[s]['composite_stopping']['final_kappa'] for s in mc_data]
mc_aucs_c = [mc_data[s]['composite_stopping']['final_auc'] for s in mc_data]

print(f"Multiclass kappa stopping:")
print(f"  Mean accuracy: {np.mean(mc_accuracies_k):.4f}")
print(f"  Mean kappa: {np.mean(mc_kappas_k):.4f}")
print(f"  Mean AUC: {np.mean(mc_aucs_k):.4f}")

print(f"Multiclass composite stopping:")
print(f"  Mean accuracy: {np.mean(mc_accuracies_c):.4f}")
print(f"  Mean kappa: {np.mean(mc_kappas_c):.4f}")
print(f"  Mean AUC: {np.mean(mc_aucs_c):.4f}")

Multiclass kappa stopping:
  Mean accuracy: 0.5038
  Mean kappa: 0.3382
  Mean AUC: 0.7088
Multiclass composite stopping:
  Mean accuracy: 0.5004
  Mean kappa: 0.3341
  Mean AUC: 0.7075


In [68]:
results_path = Path('../results')

# Binary canonical
with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    bin_data = json.load(f)

# Multiclass 16ch canonical
with open(results_path / 'loso_results_multiclass_16ch_with_checkpoints.json') as f:
    mc16_data = json.load(f)

# Multiclass 8ch canonical
with open(results_path / 'loso_results_multiclass.json') as f:
    mc8_data = json.load(f)

# Binary numbers
bin_k = [bin_data[s]['kappa_stopping']['final_kappa'] for s in bin_data]
bin_c = [bin_data[s]['composite_stopping']['final_kappa'] for s in bin_data]
bin_acc_k = [bin_data[s]['kappa_stopping']['final_accuracy'] for s in bin_data]
bin_auc_k = [bin_data[s]['kappa_stopping']['final_auc'] for s in bin_data]
bin_acc_c = [bin_data[s]['composite_stopping']['final_accuracy'] for s in bin_data]
bin_auc_c = [bin_data[s]['composite_stopping']['final_auc'] for s in bin_data]

print("BINARY 16ch:")
print(f"  Kappa stopping: mean={np.mean(bin_k):.4f}, above={sum(k>0.1 for k in bin_k)}/106")
print(f"  Composite stopping: mean={np.mean(bin_c):.4f}, above={sum(k>0.1 for k in bin_c)}/106")
print(f"  Mean accuracy kappa: {np.mean(bin_acc_k):.4f}")
print(f"  Mean AUC kappa: {np.mean(bin_auc_k):.4f}")
print(f"  Mean accuracy composite: {np.mean(bin_acc_c):.4f}")
print(f"  Mean AUC composite: {np.mean(bin_auc_c):.4f}")

# Multiclass 16ch numbers
mc16_k = [mc16_data[s]['kappa_stopping']['final_kappa'] for s in mc16_data]
mc16_c = [mc16_data[s]['composite_stopping']['final_kappa'] for s in mc16_data]
mc16_acc_k = [mc16_data[s]['kappa_stopping']['final_accuracy'] for s in mc16_data]
mc16_auc_k = [mc16_data[s]['kappa_stopping']['final_auc'] for s in mc16_data]
mc16_acc_c = [mc16_data[s]['composite_stopping']['final_accuracy'] for s in mc16_data]
mc16_auc_c = [mc16_data[s]['composite_stopping']['final_auc'] for s in mc16_data]

print("\nMULTICLASS 16ch:")
print(f"  Kappa stopping: mean={np.mean(mc16_k):.4f}, above={sum(k>0.1 for k in mc16_k)}/106")
print(f"  Composite stopping: mean={np.mean(mc16_c):.4f}, above={sum(k>0.1 for k in mc16_c)}/106")
print(f"  Mean accuracy kappa: {np.mean(mc16_acc_k):.4f}")
print(f"  Mean AUC kappa: {np.mean(mc16_auc_k):.4f}")
print(f"  Mean accuracy composite: {np.mean(mc16_acc_c):.4f}")
print(f"  Mean AUC composite: {np.mean(mc16_auc_c):.4f}")

# Per class accuracy multiclass 16ch
with open(results_path / 'per_class_accuracy' / 'multiclass_16ch_per_class_accuracy.json') as f:
    pc_data = json.load(f)

classes = {'0': 'Left hand', '1': 'Right hand', '2': 'Both fists', '3': 'Both feet'}
print("\nPER CLASS ACCURACY (multiclass 16ch, kappa stopping):")
for cls_id, cls_name in classes.items():
    accs = [pc_data[s]['kappa']['per_class_accuracy'][cls_id] for s in pc_data]
    print(f"  {cls_name}: {np.mean(accs):.4f}")

print("\nPER CLASS ACCURACY (multiclass 16ch, composite stopping):")
for cls_id, cls_name in classes.items():
    accs = [pc_data[s]['composite']['per_class_accuracy'][cls_id] for s in pc_data]
    print(f"  {cls_name}: {np.mean(accs):.4f}")

BINARY 16ch:
  Kappa stopping: mean=0.5332, above=103/106
  Composite stopping: mean=0.5307, above=104/106
  Mean accuracy kappa: 0.7661
  Mean AUC kappa: 0.7818
  Mean accuracy composite: 0.7654
  Mean AUC composite: 0.7832

MULTICLASS 16ch:
  Kappa stopping: mean=0.3415, above=97/106
  Composite stopping: mean=0.3437, above=97/106
  Mean accuracy kappa: 0.5059
  Mean AUC kappa: 0.7104
  Mean accuracy composite: 0.5082
  Mean AUC composite: 0.7108

PER CLASS ACCURACY (multiclass 16ch, kappa stopping):
  Left hand: 0.5475
  Right hand: 0.4737
  Both fists: 0.4586
  Both feet: 0.5454

PER CLASS ACCURACY (multiclass 16ch, composite stopping):
  Left hand: 0.5562
  Right hand: 0.4762
  Both fists: 0.4689
  Both feet: 0.5285


In [ ]:
mc16_k = [mc16_data[s]['kappa_stopping']['final_kappa'] for s in mc16_data]
mc16_c = [mc16_data[s]['composite_stopping']['final_kappa'] for s in mc16_data]

print("MULTICLASS 16ch kappa stopping:")
print(f"  Mean: {np.mean(mc16_k):.4f}")
print(f"  Median: {np.median(mc16_k):.4f}")
print(f"  Std: {np.std(mc16_k):.4f}")
print(f"  Above 0.1: {sum(k>0.1 for k in mc16_k)}/106")

print("\nMULTICLASS 16ch composite stopping:")
print(f"  Mean: {np.mean(mc16_c):.4f}")
print(f"  Median: {np.median(mc16_c):.4f}")
print(f"  Std: {np.std(mc16_c):.4f}")
print(f"  Above 0.1: {sum(k>0.1 for k in mc16_c)}/106")

MULTICLASS 16ch kappa stopping:
  Mean: 0.3415
  Median: 0.3237
  Std: 0.1930
  Above 0.1: 97/106

MULTICLASS 16ch composite stopping:
  Mean: 0.3437
  Median: 0.3336
  Std: 0.1955
  Above 0.1: 97/106


In [77]:
first = list(bin_data.keys())[0]
log = bin_data[first]['kappa_stopping']['log']
print(type(log))
print(list(log.keys())[:3])
first_epoch_key = list(log.keys())[0]
print(f"First epoch key: {first_epoch_key}")
print(f"First epoch data: {log[first_epoch_key]}")
last_epoch_key = list(log.keys())[-1]
print(f"Last epoch key: {last_epoch_key}")
print(f"Last epoch data: {log[last_epoch_key]}")

<class 'dict'>
['epoch', 'class_loss', 'domain_loss']
First epoch key: epoch
First epoch data: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
Last epoch key: final_confusion_matrix
Last epoch data: None


In [80]:
# check what keys are available in the log
first = list(bin_data.keys())[0]
log = bin_data[first]['kappa_stopping']['log']
print("All log keys:", list(log.keys()))

All log keys: ['epoch', 'class_loss', 'domain_loss', 'accuracy', 'kappa', 'auc', 'alpha', 'learning_rate', 'best_weights_saved', 'domain_accuracy', 'source_domain_accuracy', 'target_domain_accuracy', 'per_class_accuracy', 'confusion_matrix', 'pre_mmd_bandpower_multiscale', 'pre_mmd_bandpower_optimal', 'pre_mmd_bandpower_optimal_bandwidth', 'pre_riemannian', 'post_mmd_bandpower_multiscale', 'post_mmd_bandpower_optimal', 'post_mmd_bandpower_optimal_bandwidth', 'post_riemannian', 'post_mmd_embedding_multiscale', 'post_mmd_embedding_optimal', 'post_mmd_embedding_optimal_bandwidth', 'best_epoch', 'best_kappa', 'total_epochs_run', 'early_stopping_triggered', 'adaptation_quality', 'initial_domain_loss', 'final_domain_loss', 'final_accuracy', 'final_kappa', 'final_auc', 'final_per_class_accuracy', 'final_confusion_matrix']


In [79]:
import numpy as np

domain_acc = []
source_acc = []
target_acc = []
adaptation_quality = []
pre_mmd_multiscale = []
pre_riemannian = []
post_mmd_multiscale = []
post_mmd_embedding = []

for s in bin_data:
    log = bin_data[s]['kappa_stopping']['log']
    if log:
        domain_acc.append(log['domain_accuracy'][-1])
        source_acc.append(log['source_domain_accuracy'][-1])
        target_acc.append(log['target_domain_accuracy'][-1])
        adaptation_quality.append(log['adaptation_quality'])
        pre_mmd_multiscale.append(log['pre_mmd_bandpower_multiscale'])
        pre_riemannian.append(log['pre_riemannian'])
        post_mmd_multiscale.append(log['post_mmd_bandpower_multiscale'])
        post_mmd_embedding.append(log['post_mmd_embedding_multiscale'])

print(f"Mean final domain accuracy: {np.mean(domain_acc):.4f}")
print(f"Mean source domain accuracy: {np.mean(source_acc):.4f}")
print(f"Mean target domain accuracy: {np.mean(target_acc):.4f}")
print(f"Mean adaptation quality: {np.mean(adaptation_quality):.4f}")
print(f"Mean pre-adaptation MMD bandpower: {np.mean(pre_mmd_multiscale):.4f}")
print(f"Mean pre-adaptation Riemannian: {np.mean(pre_riemannian):.4f}")
print(f"Mean post-adaptation MMD bandpower: {np.mean(post_mmd_multiscale):.4f}")
print(f"Mean post-adaptation MMD embedding: {np.mean(post_mmd_embedding):.4f}")

Mean final domain accuracy: 0.5317
Mean source domain accuracy: 0.4584
Mean target domain accuracy: 0.6050
Mean adaptation quality: 1.0464
Mean pre-adaptation MMD bandpower: 0.2347
Mean pre-adaptation Riemannian: 2.7128
Mean post-adaptation MMD bandpower: 0.2375
Mean post-adaptation MMD embedding: 0.2062


In [ ]:
aq = [bin_data[s]['kappa_stopping']['log']['adaptation_quality'] for s in bin_data]
print(f"Adaptation quality mean: {np.mean(aq):.4f}")
print(f"Adaptation quality median: {np.median(aq):.4f}")
print(f"Below 1.0 (domain loss decreased): {sum(a < 1.0 for a in aq)}/106")
print(f"Above 1.0 (domain loss increased): {sum(a > 1.0 for a in aq)}/106")
print(f"Min: {np.min(aq):.4f}, Max: {np.max(aq):.4f}")

Adaptation quality mean: 1.0464
Adaptation quality median: 1.0474
Below 1.0 (domain loss decreased): 0/106
Above 1.0 (domain loss increased): 106/106
Min: 1.0023, Max: 1.0923


In [82]:
initial_loss = [bin_data[s]['kappa_stopping']['log']['initial_domain_loss'] for s in bin_data]
final_loss = [bin_data[s]['kappa_stopping']['log']['final_domain_loss'] for s in bin_data]

print(f"Mean initial domain loss: {np.mean(initial_loss):.4f}")
print(f"Mean final domain loss: {np.mean(final_loss):.4f}")
print(f"Subjects where final > 1.386 (discriminator below chance): {sum(f > 1.386 for f in final_loss)}/106")
print(f"Subjects where final < 1.386 (discriminator above chance): {sum(f < 1.386 for f in final_loss)}/106")

Mean initial domain loss: 2.1299
Mean final domain loss: 2.2283
Subjects where final > 1.386 (discriminator below chance): 106/106
Subjects where final < 1.386 (discriminator above chance): 0/106


In [83]:
s = 'S007'
log = bin_data[s]['kappa_stopping']['log']
epochs = log['epoch']
domain_losses = log['domain_loss']
domain_accs = log['domain_accuracy']

print("First 15 epochs:")
for i in range(min(15, len(epochs))):
    print(f"Epoch {epochs[i]}: domain_loss={domain_losses[i]:.4f}, domain_acc={domain_accs[i]:.4f}")

First 15 epochs:
Epoch 1: domain_loss=2.2013, domain_acc=0.4964
Epoch 2: domain_loss=2.3246, domain_acc=0.4996
Epoch 3: domain_loss=2.3959, domain_acc=0.5000
Epoch 4: domain_loss=2.4670, domain_acc=0.5002
Epoch 5: domain_loss=2.5072, domain_acc=0.5009
Epoch 6: domain_loss=2.5310, domain_acc=0.4998
Epoch 7: domain_loss=2.5448, domain_acc=0.5022
Epoch 8: domain_loss=2.5974, domain_acc=0.4991
Epoch 9: domain_loss=2.6029, domain_acc=0.4999
Epoch 10: domain_loss=2.5832, domain_acc=0.5003
Epoch 11: domain_loss=2.0952, domain_acc=0.6458
Epoch 12: domain_loss=2.1400, domain_acc=0.6238
Epoch 13: domain_loss=2.1383, domain_acc=0.6062
Epoch 14: domain_loss=2.1885, domain_acc=0.5931
Epoch 15: domain_loss=2.1855, domain_acc=0.5845


## 4.2

### 4.2.1

In [ ]:
base_path = '..'
lrp_binary_path = os.path.join(base_path, 'results', 'lrp_attributions', 'binary_cdan_16ch')

# Check structure of one file
test_file = os.path.join(lrp_binary_path, 'S001_lrp.npz')
data = np.load(test_file, allow_pickle=True)

print("Keys in LRP npz file:")
for k in data.keys():
    print(f"  {k}: shape {data[k].shape if hasattr(data[k], 'shape') else type(data[k])}")

Keys in LRP npz file:
  n_classes: shape ()
  labels_kappa: shape (35,)
  n_trials_kappa: shape ()
  mean_attr_class_0_kappa: shape (16, 480)
  mean_attr_class_1_kappa: shape (16, 480)
  labels_composite: shape (37,)
  n_trials_composite: shape ()
  mean_attr_class_0_composite: shape (16, 480)
  mean_attr_class_1_composite: shape (16, 480)


In [92]:
ch_idx = {ch: i for i, ch in enumerate(ch16_list)}

def get_mean_attr(subj, cls_idx, criterion='kappa'):
    data = binary_attrs_106[subj][criterion]['mean_by_class'][cls_idx]
    if data is None:
        return None
    return data.mean(axis=1)

group_order = ['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']
key_chs = ['C3', 'CP3', 'C4', 'CP4', 'CPz']

print("Score group counts:")
from collections import Counter
print(dict(Counter(score_groups.values())))
print()

for cls_idx, cls_name in [(0, 'LEFT'), (1, 'RIGHT')]:
    print(f"\n{cls_name} class — mean relevance by score group:")
    header = f"{'Group':<22}" + "".join(f"{ch:>10}" for ch in key_chs)
    print(header)
    print("-" * (22 + 10 * len(key_chs)))
    for grp in group_order:
        subjs = [s for s, g in score_groups.items() if g == grp]
        attrs = []
        for s in subjs:
            if s in binary_attrs_106:
                arr = get_mean_attr(s, cls_idx)
                if arr is not None:
                    attrs.append(arr)
        if attrs:
            mean_attr = np.mean(attrs, axis=0)
            row = f"{grp:<22}"
            for ch in key_chs:
                row += f"{mean_attr[ch_idx[ch]]:>10.6f}"
            print(row)

# Mann-Whitney: high vs low, ipsilateral relevance
print("\n\nMann-Whitney U: high vs low, mean ipsilateral relevance")

def mean_ipsi(subj, cls_idx, ipsi_chs):
    arr = get_mean_attr(subj, cls_idx)
    if arr is None:
        return None
    return np.mean([arr[ch_idx[ch]] for ch in ipsi_chs])

high_subjs = [s for s, g in score_groups.items() if g == 'high']
low_subjs  = [s for s, g in score_groups.items() if g == 'low']

for cls_idx, cls_name, ipsi_chs in [
    (0, 'LEFT',  ['C3', 'CP3']),
    (1, 'RIGHT', ['C4', 'CP4'])
]:
    high_vals = [v for s in high_subjs
                 if s in binary_attrs_106
                 and (v := mean_ipsi(s, cls_idx, ipsi_chs)) is not None]
    low_vals  = [v for s in low_subjs
                 if s in binary_attrs_106
                 and (v := mean_ipsi(s, cls_idx, ipsi_chs)) is not None]
    u, p = mannwhitneyu(high_vals, low_vals, alternative='greater')
    print(f"{cls_name}: U={u:.1f}, p={p:.4f}, "
          f"high n={len(high_vals)}, low n={len(low_vals)}")

Score group counts:
{'mid_high': 21, 'high': 30, 'negative_transfer': 3, 'low': 28, 'mid_low': 24}


LEFT class — mean relevance by score group:
Group                         C3       CP3        C4       CP4       CPz
------------------------------------------------------------------------
high                    0.001934  0.002259 -0.001266 -0.001225 -0.000004
mid_high                0.001806  0.001903 -0.001246 -0.001246 -0.000022
mid_low                 0.001634  0.001831 -0.001187 -0.001176  0.000062
low                     0.001475  0.001578 -0.001267 -0.001348  0.000048
negative_transfer       0.000926  0.001156 -0.000924 -0.000868  0.000192

RIGHT class — mean relevance by score group:
Group                         C3       CP3        C4       CP4       CPz
------------------------------------------------------------------------
high                   -0.001261 -0.001293  0.002333  0.002109 -0.000145
mid_high               -0.001259 -0.001349  0.002067  0.002047 -0.000129
mid_lo

In [93]:
category_order = ['neither', 'group_a_collapse', 'group_b_flat', 'negative_transfer']
key_chs = ['C3', 'CP3', 'C4', 'CP4', 'CPz']

print("Category group counts:")
from collections import Counter
print(dict(Counter(category_groups_106.values())))
print()

for cls_idx, cls_name in [(0, 'LEFT'), (1, 'RIGHT')]:
    print(f"\n{cls_name} class — mean LRP relevance by category group:")
    header = f"{'Group':<22}" + "".join(f"{ch:>10}" for ch in key_chs)
    print(header)
    print("-" * (22 + 10 * len(key_chs)))
    for grp in category_order:
        subjs = [s for s, g in category_groups_106.items() if g == grp]
        attrs = []
        for s in subjs:
            if s in binary_attrs_106:
                arr = get_mean_attr(s, cls_idx)
                if arr is not None:
                    attrs.append(arr)
        if attrs:
            mean_attr = np.mean(attrs, axis=0)
            row = f"{grp:<22}"
            for ch in key_chs:
                row += f"{mean_attr[ch_idx[ch]]:>10.6f}"
            print(row)

Category group counts:
{'neither': 87, 'negative_transfer': 3, 'group_a_collapse': 13, 'group_b_flat': 3}


LEFT class — mean LRP relevance by category group:
Group                         C3       CP3        C4       CP4       CPz
------------------------------------------------------------------------
neither                 0.001836  0.002025 -0.001313 -0.001285  0.000020
group_a_collapse        0.001162  0.001413 -0.000959 -0.001243  0.000035
group_b_flat            0.000541  0.000453 -0.000476 -0.000304  0.000019
negative_transfer       0.000926  0.001156 -0.000924 -0.000868  0.000192

RIGHT class — mean LRP relevance by category group:
Group                         C3       CP3        C4       CP4       CPz
------------------------------------------------------------------------
neither                -0.001323 -0.001320  0.002067  0.002050 -0.000151
group_a_collapse       -0.000860 -0.000872  0.001090  0.001431 -0.000052
group_b_flat           -0.000604 -0.000344  0.000616  0.00

##### Debug

In [ ]:
import os
pc_file = per_class_path / "multiclass_16ch_per_class_accuracy.json"
print("File path:", pc_file)
print("Last modified:", os.path.getmtime(pc_file))

import datetime
mtime = os.path.getmtime(pc_file)
print("Modified date:", datetime.datetime.fromtimestamp(mtime))

In [95]:
import json

# check a few subjects that should have changed category
# state doc says group_b went from 3 to 7 subjects
# check what subjects are currently in group_b_flat
group_b_subjects = [s for s, g in category_groups_106.items() if g == 'group_b_flat']
print("Current group_b subjects:", sorted(group_b_subjects))

group_a_subjects = [s for s, g in category_groups_106.items() if g == 'group_a_collapse']
print("Current group_a subjects:", sorted(group_a_subjects))

# check which multiclass file per_class_accuracy was generated from
# by looking at a subject that should have moved between groups
# state doc group_b new: S020 S027 S057 S058 S066 S087 S089
for subj in ['S027', 'S057', 'S058', 'S087']:
    pca = all_per_class[subj]
    print(f"{subj} per_class_accuracy: {pca}")

Current group_b subjects: ['S020', 'S066', 'S089']
Current group_a subjects: ['S006', 'S014', 'S016', 'S017', 'S021', 'S024', 'S037', 'S038', 'S047', 'S067', 'S081', 'S086', 'S099']
S027 per_class_accuracy: {0: 0.4782608695652174, 1: 0.36363636363636365, 2: 0.5, 3: 0.30434782608695654}
S057 per_class_accuracy: {0: 0.4583333333333333, 1: 0.5238095238095238, 2: 0.45454545454545453, 3: 0.43478260869565216}
S058 per_class_accuracy: {0: 0.5, 1: 0.47619047619047616, 2: 0.34782608695652173, 3: 0.5454545454545454}
S087 per_class_accuracy: {0: 0.42857142857142855, 1: 0.45454545454545453, 2: 0.34782608695652173, 3: 0.47619047619047616}


In [96]:
# Check the thresholds classify_failure_groups is actually using
import inspect
print(inspect.getsource(classify_failure_groups))
print()

# Check binary kappas for the subjects that should be in group_b
# according to state doc but are not showing up there now
check_subjects = ['S027', 'S057', 'S058', 'S087', 'S067']
print("Binary kappas for subjects of interest:")
for subj in check_subjects:
    k = all_kappas[subj]
    cat = category_groups_106[subj]
    pca = all_per_class[subj]
    spread = max(pca.values()) - min(pca.values())
    print(f"{subj}: kappa={k:.4f}, category={cat}, "
          f"spread={spread:.4f}, min_acc={min(pca.values()):.4f}")

def classify_failure_groups(
    all_per_class,
    all_kappas,
    spread_threshold=0.4,
    min_acc_threshold=0.2,
    flat_spread_threshold=0.2,
    flat_kappa_threshold=0.3,
):
    """
    Classifies subjects into failure groups based on per-class accuracy.

    Group A (selective collapse): spread > spread_threshold AND
        min class accuracy < min_acc_threshold.
        One class is substantially worse than others.

    Group B (flat near-chance): spread < flat_spread_threshold AND
        overall kappa < flat_kappa_threshold.
        All classes performing similarly poorly.

    Neither: everything else.

    Thresholds derived from full 106-subject distribution:
        spread mean=0.343, std=0.145
        min_acc mean=0.332, std=0.165
    """
    group_a = []
    group_b = []
    neither = []

    for subj in all_per_class:
        pca = all_per_class[subj]
        spread = max(pca.values()) - min(pca.values())
        min_acc = min(pca.values())
        kappa = all_kappas

In [97]:
print("NEGATIVE TRANSFER subjects:")
nt = [s for s, g in category_groups_106.items() if g == 'negative_transfer']
print(sorted(nt))
print()

print("GROUP A subjects:")
ga = [s for s, g in category_groups_106.items() if g == 'group_a_collapse']
print(sorted(ga))
print(f"Count: {len(ga)}")
print()

print("GROUP B subjects:")
gb = [s for s, g in category_groups_106.items() if g == 'group_b_flat']
print(sorted(gb))
print(f"Count: {len(gb)}")
print()

print("NEITHER count:", len([s for s, g in category_groups_106.items() if g == 'neither']))

NEGATIVE TRANSFER subjects:
['S003', 'S005', 'S018']

GROUP A subjects:
['S006', 'S014', 'S016', 'S017', 'S021', 'S024', 'S037', 'S038', 'S047', 'S067', 'S081', 'S086', 'S099']
Count: 13

GROUP B subjects:
['S020', 'S066', 'S089']
Count: 3

NEITHER count: 87


### 4.2.2

In [98]:
category_order = ['neither', 'group_a_collapse', 'group_b_flat', 'negative_transfer']
key_chs = ['C3', 'CP3', 'C4', 'CP4', 'CPz']

for cls_idx, cls_name in [(0, 'LEFT'), (1, 'RIGHT')]:
    print(f"\n{cls_name} class — mean LRP relevance by category group:")
    header = f"{'Group':<22}" + "".join(f"{ch:>10}" for ch in key_chs)
    print(header)
    print("-" * (22 + 10 * len(key_chs)))
    for grp in category_order:
        subjs = [s for s, g in category_groups_106.items() if g == grp]
        attrs = []
        for s in subjs:
            if s in binary_attrs_106:
                arr = get_mean_attr(s, cls_idx)
                if arr is not None:
                    attrs.append(arr)
        if attrs:
            mean_attr = np.mean(attrs, axis=0)
            row = f"{grp:<22}"
            for ch in key_chs:
                row += f"{mean_attr[ch_idx[ch]]:>10.6f}"
            print(row)

# Group B detailed: individual subject attribution strength
print("\n\nGroup B individual subjects — mean ipsilateral relevance:")
print(f"{'Subject':<10} {'Left_C3':>10} {'Left_CP3':>10} {'Right_C4':>10} {'Right_CP4':>10} {'Binary_kappa':>14}")
for s in sorted([s for s, g in category_groups_106.items() if g == 'group_b_flat']):
    if s in binary_attrs_106:
        left = get_mean_attr(s, 0)
        right = get_mean_attr(s, 1)
        k = all_kappas[s]
        print(f"{s:<10} {left[ch_idx['C3']]:>10.6f} {left[ch_idx['CP3']]:>10.6f} "
              f"{right[ch_idx['C4']]:>10.6f} {right[ch_idx['CP4']]:>10.6f} {k:>14.4f}")


LEFT class — mean LRP relevance by category group:
Group                         C3       CP3        C4       CP4       CPz
------------------------------------------------------------------------
neither                 0.001836  0.002025 -0.001313 -0.001285  0.000020
group_a_collapse        0.001162  0.001413 -0.000959 -0.001243  0.000035
group_b_flat            0.000541  0.000453 -0.000476 -0.000304  0.000019
negative_transfer       0.000926  0.001156 -0.000924 -0.000868  0.000192

RIGHT class — mean LRP relevance by category group:
Group                         C3       CP3        C4       CP4       CPz
------------------------------------------------------------------------
neither                -0.001323 -0.001320  0.002067  0.002050 -0.000151
group_a_collapse       -0.000860 -0.000872  0.001090  0.001431 -0.000052
group_b_flat           -0.000604 -0.000344  0.000616  0.000393  0.000004
negative_transfer      -0.001792 -0.001412  0.002048  0.001864 -0.000154


Group B individua

### 4.2.3

In [99]:
# Temporal profiles for key channels by score group
# Average attribution over subjects, then look at time dimension
# for C3/CP3 (left class) and C4/CP4 (right class)

group_order = ['high', 'low', 'negative_transfer']
time_axis = np.arange(480) / 160  # convert to seconds at 160Hz

print("Temporal profile check — mean attribution over time")
print("Left class, CP3 (index 5):")
for grp in group_order:
    subjs = [s for s, g in score_groups.items() if g == grp]
    attrs = []
    for s in subjs:
        if s in binary_attrs_106:
            arr = binary_attrs_106[s]['kappa']['mean_by_class'][0]
            if arr is not None:
                attrs.append(arr[5, :])  # CP3 time series
    if attrs:
        mean_ts = np.mean(attrs, axis=0)
        # find peak window
        peak_idx = np.argmax(mean_ts)
        peak_time = time_axis[peak_idx]
        # mean in 0.3-1.0s window vs 1.0-3.0s window
        early = mean_ts[(time_axis >= 0.3) & (time_axis < 1.0)].mean()
        late = mean_ts[(time_axis >= 1.0) & (time_axis <= 3.0)].mean()
        print(f"  {grp}: peak at {peak_time:.2f}s, "
              f"early(0.3-1.0s)={early:.6f}, late(1.0-3.0s)={late:.6f}")

print("\nRight class, C4 (index 2):")
for grp in group_order:
    subjs = [s for s, g in score_groups.items() if g == grp]
    attrs = []
    for s in subjs:
        if s in binary_attrs_106:
            arr = binary_attrs_106[s]['kappa']['mean_by_class'][1]
            if arr is not None:
                attrs.append(arr[2, :])  # C4 time series
    if attrs:
        mean_ts = np.mean(attrs, axis=0)
        peak_idx = np.argmax(mean_ts)
        peak_time = time_axis[peak_idx]
        early = mean_ts[(time_axis >= 0.3) & (time_axis < 1.0)].mean()
        late = mean_ts[(time_axis >= 1.0) & (time_axis <= 3.0)].mean()
        print(f"  {grp}: peak at {peak_time:.2f}s, "
              f"early(0.3-1.0s)={early:.6f}, late(1.0-3.0s)={late:.6f}")

# SHAP temporal profile for comparison
print("\nSHAP temporal profile — Left class, CP3:")
for grp in ['high', 'low']:
    subjs = [s for s, g in score_groups.items() if g == grp]
    attrs = []
    for s in subjs:
        if s in binary_shap_106:
            arr = binary_shap_106[s]['kappa']['mean_by_class'][0]
            if arr is not None:
                attrs.append(arr[5, :])
    if attrs:
        mean_ts = np.mean(attrs, axis=0)
        early = mean_ts[(time_axis >= 0.3) & (time_axis < 0.7)].mean()
        late = mean_ts[(time_axis >= 0.7) & (time_axis <= 3.0)].mean()
        print(f"  {grp}: early(0.3-0.7s)={early:.6f}, late(0.7-3.0s)={late:.6f}")

Temporal profile check — mean attribution over time
Left class, CP3 (index 5):
  high: peak at 0.53s, early(0.3-1.0s)=0.004304, late(1.0-3.0s)=0.001729
  low: peak at 0.49s, early(0.3-1.0s)=0.002970, late(1.0-3.0s)=0.001211
  negative_transfer: peak at 0.54s, early(0.3-1.0s)=0.002247, late(1.0-3.0s)=0.000788

Right class, C4 (index 2):
  high: peak at 0.65s, early(0.3-1.0s)=0.003616, late(1.0-3.0s)=0.002186
  low: peak at 0.63s, early(0.3-1.0s)=0.002539, late(1.0-3.0s)=0.001202
  negative_transfer: peak at 0.93s, early(0.3-1.0s)=0.002861, late(1.0-3.0s)=0.001997

SHAP temporal profile — Left class, CP3:
  high: early(0.3-0.7s)=0.000994, late(0.7-3.0s)=0.000209
  low: early(0.3-0.7s)=0.000015, late(0.7-3.0s)=-0.000121


In [100]:
# Group B temporal profile vs neither for comparison
print("Temporal profile — Left class CP3, category groups:")
print(f"{'Group':<20} {'Peak(s)':>10} {'Early 0.3-1.0s':>16} {'Late 1.0-3.0s':>14}")

for grp, grp_dict in [('neither', category_groups_106),
                       ('group_b_flat', category_groups_106)]:
    subjs = [s for s, g in grp_dict.items() if g == grp]
    attrs = []
    for s in subjs:
        if s in binary_attrs_106:
            arr = binary_attrs_106[s]['kappa']['mean_by_class'][0]
            if arr is not None:
                attrs.append(arr[5, :])  # CP3
    if attrs:
        mean_ts = np.mean(attrs, axis=0)
        peak_idx = np.argmax(mean_ts)
        peak_time = time_axis[peak_idx]
        early = mean_ts[(time_axis >= 0.3) & (time_axis < 1.0)].mean()
        late = mean_ts[(time_axis >= 1.0) & (time_axis <= 3.0)].mean()
        print(f"{grp:<20} {peak_time:>10.2f} {early:>16.6f} {late:>14.6f}")

# Individual Group B subjects temporal variability
print("\nGroup B individual subjects — CP3 left class temporal std (burstiness indicator):")
for s in sorted([s for s, g in category_groups_106.items() if g == 'group_b_flat']):
    if s in binary_attrs_106:
        arr = binary_attrs_106[s]['kappa']['mean_by_class'][0]
        if arr is not None:
            ts = arr[5, :]
            early_std = ts[(time_axis >= 0.3) & (time_axis < 1.0)].std()
            late_std = ts[(time_axis >= 1.0) & (time_axis <= 3.0)].std()
            print(f"  {s}: early_std={early_std:.6f}, late_std={late_std:.6f}")

Temporal profile — Left class CP3, category groups:
Group                   Peak(s)   Early 0.3-1.0s  Late 1.0-3.0s
neither                    0.52         0.003900       0.001512
group_b_flat               1.26         0.000598       0.000422

Group B individual subjects — CP3 left class temporal std (burstiness indicator):
  S020: early_std=0.000397, late_std=0.000308
  S066: early_std=0.000295, late_std=0.000175
  S089: early_std=0.000805, late_std=0.000900


### 4.2.4

In [102]:
# Compare kappa vs composite attribution maps for key channels
# Correlation between kappa and composite stopping attributions per subject
# then report mean correlation across all 106 subjects

from scipy.stats import pearsonr

print("Kappa vs composite stopping — attribution correlation per subject")
print("Left class CP3, Right class C4")
print()

corr_left = []
corr_right = []

for s in valid_subjects:
    if s in binary_attrs_106:
        left_k = binary_attrs_106[s]['kappa']['mean_by_class'][0]
        left_c = binary_attrs_106[s]['composite']['mean_by_class'][0]
        right_k = binary_attrs_106[s]['kappa']['mean_by_class'][1]
        right_c = binary_attrs_106[s]['composite']['mean_by_class'][1]

        if left_k is not None and left_c is not None:
            r_left, _ = pearsonr(left_k.flatten(), left_c.flatten())
            corr_left.append(r_left)
        if right_k is not None and right_c is not None:
            r_right, _ = pearsonr(right_k.flatten(), right_c.flatten())
            corr_right.append(r_right)

print(f"Left class: mean r={np.mean(corr_left):.4f}, "
      f"std={np.std(corr_left):.4f}, "
      f"min={np.min(corr_left):.4f}")
print(f"Right class: mean r={np.mean(corr_right):.4f}, "
      f"std={np.std(corr_right):.4f}, "
      f"min={np.min(corr_right):.4f}")
print(f"Subjects compared: {len(corr_left)}")

Kappa vs composite stopping — attribution correlation per subject
Left class CP3, Right class C4

Left class: mean r=0.4607, std=0.1542, min=-0.0183
Right class: mean r=0.4689, std=0.1419, min=0.0759
Subjects compared: 106


### 4.3.3

In [103]:
time_axis = np.arange(480) / 160

print("Multiclass temporal profile — early vs late window")
print("Using neither group subjects only")
print()

neither_subjs = [s for s, g in category_groups_106.items() if g == 'neither']

class_names = ['left', 'right', 'both_fists', 'both_feet']
key_ch_per_class = [5, 2, 6, 0]  # CP3, C4, CPz, C3

for cls_idx, (cls_name, ch_idx_key) in enumerate(zip(class_names, key_ch_per_class)):
    attrs = []
    for s in neither_subjs:
        if s in mc_attrs_106:
            arr = mc_attrs_106[s]['kappa']['mean_by_class'][cls_idx]
            if arr is not None:
                attrs.append(arr[ch_idx_key, :])
    if attrs:
        mean_ts = np.mean(attrs, axis=0)
        peak_idx = np.argmax(mean_ts)
        peak_time = time_axis[peak_idx]
        early = mean_ts[(time_axis >= 0.3) & (time_axis < 1.0)].mean()
        late = mean_ts[(time_axis >= 1.0) & (time_axis <= 3.0)].mean()
        print(f"{cls_name} ({ch16_list[ch_idx_key]}): "
              f"peak={peak_time:.2f}s, "
              f"early(0.3-1.0s)={early:.6f}, "
              f"late(1.0-3.0s)={late:.6f}")

Multiclass temporal profile — early vs late window
Using neither group subjects only

left (CP3): peak=0.51s, early(0.3-1.0s)=0.004155, late(1.0-3.0s)=0.001488
right (C4): peak=0.50s, early(0.3-1.0s)=0.003826, late(1.0-3.0s)=0.002865
both_fists (CPz): peak=1.30s, early(0.3-1.0s)=0.001104, late(1.0-3.0s)=0.002355
both_feet (C3): peak=1.05s, early(0.3-1.0s)=0.002976, late(1.0-3.0s)=0.003375


In [104]:
print("Multiclass temporal verification — all three methods, neither group")
print("Early window 0.3-1.0s vs Late window 1.0-3.0s ratio")
print()

method_data = {
    'LRP': mc_attrs_106,
    'DeepLIFT': mc_dl_106,
    'IG': mc_ig_106
}

class_names = ['left', 'right', 'both_fists', 'both_feet']
key_ch_per_class = [5, 2, 6, 0]  # CP3, C4, CPz, C3

neither_subjs = [s for s, g in category_groups_106.items() if g == 'neither']

for method_name, method_dict in method_data.items():
    print(f"\n{method_name}:")
    print(f"{'Class':<15} {'Key Ch':<8} {'Peak(s)':>8} {'Early':>12} {'Late':>12} {'Ratio E/L':>10}")
    print("-" * 65)
    for cls_idx, (cls_name, ch_idx_key) in enumerate(
            zip(class_names, key_ch_per_class)):
        attrs = []
        for s in neither_subjs:
            if s in method_dict:
                arr = method_dict[s]['kappa']['mean_by_class'][cls_idx]
                if arr is not None:
                    attrs.append(arr[ch_idx_key, :])
        if attrs:
            mean_ts = np.mean(attrs, axis=0)
            peak_idx = np.argmax(mean_ts)
            peak_time = time_axis[peak_idx]
            early = mean_ts[(time_axis >= 0.3) & (time_axis < 1.0)].mean()
            late = mean_ts[(time_axis >= 1.0) & (time_axis <= 3.0)].mean()
            ratio = early / late if late != 0 else float('inf')
            print(f"{cls_name:<15} {ch16_list[ch_idx_key]:<8} "
                  f"{peak_time:>8.2f} {early:>12.6f} "
                  f"{late:>12.6f} {ratio:>10.3f}")

Multiclass temporal verification — all three methods, neither group
Early window 0.3-1.0s vs Late window 1.0-3.0s ratio


LRP:
Class           Key Ch    Peak(s)        Early         Late  Ratio E/L
-----------------------------------------------------------------
left            CP3          0.51     0.004155     0.001488      2.793
right           C4           0.50     0.003826     0.002865      1.335
both_fists      CPz          1.30     0.001104     0.002355      0.469
both_feet       C3           1.05     0.002976     0.003375      0.882

DeepLIFT:
Class           Key Ch    Peak(s)        Early         Late  Ratio E/L
-----------------------------------------------------------------
left            CP3          0.51     0.005222     0.002038      2.563
right           C4           0.68     0.004516     0.003099      1.457
both_fists      CPz          1.30     0.001340     0.002757      0.486
both_feet       C3           1.11     0.002940     0.003518      0.836

IG:
Class          

In [106]:
print("Multiclass temporal verification — all four classes")
print("Early vs late window ratio across all category groups")
print("LRP only for group comparison")
print()

groups = ['neither', 'group_a_collapse', 'group_b_flat', 'negative_transfer']

class_info = [
    (0, 'left', 5),        # CP3
    (1, 'right', 2),       # C4
    (2, 'both_fists', 6),  # CPz
    (3, 'both_feet', 0)    # C3
]

for cls_idx, cls_name, ch_idx_key in class_info:
    print(f"\n{cls_name} — {ch16_list[ch_idx_key]}:")
    print(f"{'Group':<22} {'Peak(s)':>8} {'Early':>12} {'Late':>12} {'Ratio':>8}")
    print("-" * 65)
    for grp in groups:
        subjs = [s for s, g in category_groups_106.items() if g == grp]
        attrs = []
        for s in subjs:
            if s in mc_attrs_106:
                arr = mc_attrs_106[s]['kappa']['mean_by_class'][cls_idx]
                if arr is not None:
                    attrs.append(arr[ch_idx_key, :])
        if attrs:
            mean_ts = np.mean(attrs, axis=0)
            peak_idx = np.argmax(mean_ts)
            peak_time = time_axis[peak_idx]
            early = mean_ts[(time_axis >= 0.3) & (time_axis < 1.0)].mean()
            late = mean_ts[(time_axis >= 1.0) & (time_axis <= 3.0)].mean()
            ratio = early / late if late != 0 else float('inf')
            print(f"{grp:<22} {peak_time:>8.2f} "
                  f"{early:>12.6f} {late:>12.6f} {ratio:>8.3f}")

Multiclass temporal verification — all four classes
Early vs late window ratio across all category groups
LRP only for group comparison


left — CP3:
Group                   Peak(s)        Early         Late    Ratio
-----------------------------------------------------------------
neither                    0.51     0.004155     0.001488    2.793
group_a_collapse           0.72     0.003430     0.001461    2.348
group_b_flat               0.84     0.004693     0.001593    2.946
negative_transfer          0.72     0.002405     0.000916    2.625

right — C4:
Group                   Peak(s)        Early         Late    Ratio
-----------------------------------------------------------------
neither                    0.50     0.003826     0.002865    1.335
group_a_collapse           0.82     0.002830     0.002051    1.380
group_b_flat               1.61     0.004058     0.003807    1.066
negative_transfer          0.80     0.003326     0.002582    1.288

both_fists — CPz:
Group           

##### complete 

In [111]:
print("=" * 70)
print("Channel-level temporal pattern")
print("Per class, per channel, across groups")
print("=" * 70)

group_labels = {
    'neither': '2a: Neither',
    'group_a_collapse': '2b: Group A',
    'group_b_flat': '2c: Group B (n=7)',
    'negative_transfer': '2d: Neg Transfer (n=3)'
}

for grp, grp_label in group_labels.items():
    print(f"\n{grp_label}")
    subjs = [s for s, g in category_groups_106.items() if g == grp]

    print(f"{'Channel':<8}", end="")
    for cls_idx, cls_name in class_info:
        print(f" {cls_name+' ratio':>14}", end="")
    print(f" {'left peak':>10} {'right peak':>10} "
          f"{'fists peak':>10} {'feet peak':>10}")
    print("-" * 90)

    for ch_i, ch_name in enumerate(ch16_list):
        row = f"{ch_name:<8}"
        peaks = []
        for cls_idx, cls_name in class_info:
            attrs = []
            for s in subjs:
                if s in mc_attrs_106:
                    arr = mc_attrs_106[s]['kappa']['mean_by_class'][cls_idx]
                    if arr is not None:
                        attrs.append(arr[ch_i, :])
            if attrs:
                mean_ts = np.mean(attrs, axis=0)
                early = mean_ts[(time_axis >= 0.3) & (time_axis < 1.0)].mean()
                late = mean_ts[(time_axis >= 1.0) & (time_axis <= 3.0)].mean()
                ratio = early / late if late != 0 else float('inf')
                peak_idx = np.argmax(np.abs(mean_ts))
                peak_time = time_axis[peak_idx]
                row += f" {ratio:>14.3f}"
                peaks.append(f"{peak_time:>10.2f}")
            else:
                row += f" {'N/A':>14}"
                peaks.append(f"{'N/A':>10}")
        print(row + " " + " ".join(peaks))

Channel-level temporal pattern
Per class, per channel, across groups

2a: Neither
Channel      left ratio    right ratio both_fists ratio both_feet ratio  left peak right peak fists peak  feet peak
------------------------------------------------------------------------------------------
C3                2.501          1.480          0.694          0.882       0.51       0.86       1.03       1.05
Cz                3.242         -2.822          0.545          1.268       0.81       0.48       1.17       1.11
C4                1.775          1.335         -0.010          1.248       0.77       0.50       1.11       0.94
FC3               6.299         -0.367          0.568          1.212       0.98       1.24       0.92       0.92
FCz               1.693          2.830          0.956          1.384       0.69       0.68       2.54       1.09
CP3               2.793          1.538          0.740          0.920       0.51       0.78       1.07       1.06
CPz               2.387          

In [113]:
# DataFrame: channel level per group
rows_out2 = []
for grp in groups:
    subjs = [s for s, g in category_groups_106.items() if g == grp]
    for ch_i, ch_name in enumerate(ch16_list):
        row = {'group': grp, 'n_subjects': len(subjs), 'channel': ch_name}
        for cls_idx, cls_name in class_info:
            attrs = []
            for s in subjs:
                if s in mc_attrs_106:
                    arr = mc_attrs_106[s]['kappa']['mean_by_class'][cls_idx]
                    if arr is not None:
                        attrs.append(arr[ch_i, :])
            if attrs:
                mean_ts = np.mean(attrs, axis=0)
                early = mean_ts[(time_axis >= 0.3) & (time_axis < 1.0)].mean()
                late = mean_ts[(time_axis >= 1.0) & (time_axis <= 3.0)].mean()
                ratio = early / late if late != 0 else float('inf')
                peak_time = time_axis[np.argmax(np.abs(mean_ts))]
                row[f'{cls_name}_ratio'] = round(ratio, 3)
                row[f'{cls_name}_peak'] = round(peak_time, 2)
        rows_out2.append(row)

df_output2 = pd.DataFrame(rows_out2)
# print("OUTPUT 2 DataFrame:")
# print(df_output2.to_string(index=False))

In [114]:
with pd.option_context('display.max_rows', 100):
    display(df_output2)

,group,n_subjects,channel,left_ratio,left_peak,right_ratio,right_peak,both_fists_ratio,both_fists_peak,both_feet_ratio,both_feet_peak
0,neither,87,C3,2.501000,0.51,1.480000,0.86,0.694,1.03,0.882,1.05
1,neither,87,Cz,3.242000,0.81,-2.822000,0.48,0.545,1.17,1.268,1.11
2,neither,87,C4,1.775000,0.77,1.335000,0.50,-0.010,1.11,1.248,0.94
3,neither,87,FC3,6.299000,0.98,-0.367000,1.24,0.568,0.92,1.212,0.92
4,neither,87,FCz,1.693000,0.69,2.830000,0.68,0.956,2.54,1.384,1.09
5,neither,87,CP3,2.793000,0.51,1.538000,0.78,0.740,1.08,0.920,1.06
6,neither,87,CPz,2.387000,0.93,3.000000,0.70,0.469,1.30,1.106,0.92
7,neither,87,CP4,2.091000,0.60,1.712000,0.50,0.007,1.10,1.469,0.94
8,neither,87,CP5,4.734000,0.59,2.175000,0.52,2.044,0.91,1.541,1.24
9,neither,87,CP6,2.161000,0.90,2.450000,0.44,0.425,1.13,1.228,0.73


In [ ]:
export_path = os.path.join(base_path, 'results', 'temporal_verification_output2.csv')
df_output2.to_csv(export_path, index=False)
print(f"Saved to {export_path}")

In [117]:
group_labels = {
    'neither': 'Neither (n=87)',
    'group_a_collapse': 'Group A (n=13)',
    'group_b_flat': 'Group B (n=7)',
    'negative_transfer': 'Negative Transfer (n=3)'
}

col_order = ['channel',
             'left_ratio', 'left_peak',
             'right_ratio', 'right_peak',
             'both_fists_ratio', 'both_fists_peak',
             'both_feet_ratio', 'both_feet_peak']

for grp, grp_label in group_labels.items():
    df_grp = df_output2[df_output2['group'] == grp][col_order].copy()
    df_grp = df_grp.set_index('channel')
    df_grp.index.name = None
    print(f"\n{grp_label}")
    with pd.option_context('display.max_rows', 20,
                           'display.max_columns', 20,
                           'display.float_format', '{:.3f}'.format):
        display(df_grp)


Neither (n=87)


,left_ratio,left_peak,right_ratio,right_peak,both_fists_ratio,both_fists_peak,both_feet_ratio,both_feet_peak
C3,2.501,0.510,1.480,0.860,0.694,1.030,0.882,1.050
Cz,3.242,0.810,-2.822,0.480,0.545,1.170,1.268,1.110
C4,1.775,0.770,1.335,0.500,-0.010,1.110,1.248,0.940
FC3,6.299,0.980,-0.367,1.240,0.568,0.920,1.212,0.920
FCz,1.693,0.690,2.830,0.680,0.956,2.540,1.384,1.090
CP3,2.793,0.510,1.538,0.780,0.740,1.080,0.920,1.060
CPz,2.387,0.930,3.000,0.700,0.469,1.300,1.106,0.920
CP4,2.091,0.600,1.712,0.500,0.007,1.100,1.469,0.940
CP5,4.734,0.590,2.175,0.520,2.044,0.910,1.541,1.240
CP6,2.161,0.900,2.450,0.440,0.425,1.130,1.228,0.730



Group A (n=13)


,left_ratio,left_peak,right_ratio,right_peak,both_fists_ratio,both_fists_peak,both_feet_ratio,both_feet_peak
C3,1.918,0.900,1.420,0.960,0.642,0.780,1.145,0.670
Cz,2.662,0.760,2.980,0.580,0.378,0.640,1.321,0.830
C4,1.913,0.720,1.380,0.820,0.113,1.140,1.249,0.920
FC3,0.196,0.620,2.380,1.810,0.501,0.790,1.354,0.670
FCz,5.448,0.910,0.963,0.770,0.291,1.140,2.004,0.670
CP3,2.348,0.720,1.731,0.860,0.651,0.780,1.183,0.890
CPz,2.809,1.210,1.686,2.100,0.477,0.780,1.203,0.980
CP4,1.879,0.910,1.760,0.810,0.049,0.140,1.362,0.780
CP5,-2.084,0.960,3.146,0.440,5.188,0.350,2.159,0.960
CP6,1.303,1.010,1.555,2.170,-0.421,2.700,1.174,1.050



Group B (n=7)


,left_ratio,left_peak,right_ratio,right_peak,both_fists_ratio,both_fists_peak,both_feet_ratio,both_feet_peak
C3,3.321,0.930,1.447,0.610,0.799,1.140,1.423,0.840
Cz,3.853,0.410,4.467,0.410,1.255,0.990,1.805,0.620
C4,2.022,0.810,1.066,1.610,0.125,1.200,0.914,0.910
FC3,-0.626,1.050,-0.595,1.730,0.702,0.650,1.123,1.210
FCz,4.572,0.890,0.836,0.950,-1.816,0.980,0.942,1.020
CP3,2.946,0.840,1.361,1.940,0.764,1.140,1.257,1.080
CPz,2.114,1.510,-25.661,0.400,0.828,0.920,1.359,0.930
CP4,2.062,1.050,1.346,1.610,-0.007,0.740,1.200,2.660
CP5,2.022,0.910,15.454,0.440,0.102,1.510,2.725,0.890
CP6,3.421,0.770,1.632,0.920,1.496,0.880,1.033,1.130



Negative Transfer (n=3)


,left_ratio,left_peak,right_ratio,right_peak,both_fists_ratio,both_fists_peak,both_feet_ratio,both_feet_peak
C3,2.746,0.560,1.388,1.200,0.661,1.120,0.823,1.080
Cz,3.001,0.880,1.091,2.770,0.235,1.950,1.682,1.150
C4,1.815,0.690,1.288,0.800,0.201,0.220,1.774,0.930
FC3,-5.210,0.490,1.437,0.640,1.736,0.880,1.227,1.180
FCz,0.786,0.650,-9.939,0.810,0.086,1.100,-1.125,0.940
CP3,2.625,0.720,1.714,0.680,0.644,1.110,0.929,1.180
CPz,6.615,0.590,0.693,1.840,0.403,1.040,0.950,1.020
CP4,2.427,0.710,2.217,0.460,0.345,0.190,2.148,0.900
CP5,1.519,0.910,0.307,0.890,2.974,1.060,0.668,0.910
CP6,3.418,0.640,2.477,0.570,-0.273,1.080,1.345,0.750


In [ ]:
# Save all four groups to one CSV with group label as first column
dfs = []
for grp, grp_label in group_labels.items():
    df_grp = df_output2[df_output2['group'] == grp][col_order].copy()
    df_grp.insert(0, 'group_label', grp_label)
    dfs.append(df_grp)

df_all_groups = pd.concat(dfs, ignore_index=True)

export_path = os.path.join(base_path, 'results',
                           'temporal_verification_output2_by_group.csv')
df_all_groups.to_csv(export_path, index=False)
print(f"Saved to {export_path}")

### Method agreement

In [121]:
# Remaining binary pairs
print("DeepLIFT vs IG agreement, binary, 106 subjects")
print("-" * 50)
agreement_dl_ig = compute_method_agreement(
    binary_dl_106, binary_ig_106,
    valid_subjects, criterion='kappa', n_classes=2
)
for cls, name in enumerate(['left', 'right']):
    correlations = [agreement_dl_ig[s][cls] for s in valid_subjects
                    if agreement_dl_ig.get(s, {}).get(cls) is not None]
    print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
          f"std={np.std(correlations):.4f}")

print()
print("DeepLIFT vs SHAP agreement, binary, 106 subjects")
print("-" * 50)
agreement_dl_shap = compute_method_agreement(
    binary_dl_106, binary_shap_106,
    valid_subjects, criterion='kappa', n_classes=2
)
for cls, name in enumerate(['left', 'right']):
    correlations = [agreement_dl_shap[s][cls] for s in valid_subjects
                    if agreement_dl_shap.get(s, {}).get(cls) is not None]
    print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
          f"std={np.std(correlations):.4f}")

print()
print("SHAP vs IG agreement, binary, 106 subjects")
print("-" * 50)
agreement_shap_ig = compute_method_agreement(
    binary_shap_106, binary_ig_106,
    valid_subjects, criterion='kappa', n_classes=2
)
for cls, name in enumerate(['left', 'right']):
    correlations = [agreement_shap_ig[s][cls] for s in valid_subjects
                    if agreement_shap_ig.get(s, {}).get(cls) is not None]
    print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
          f"std={np.std(correlations):.4f}")

print()
print("=" * 50)
print("MULTICLASS — all six pairs, all four classes")
print("=" * 50)

mc_method_pairs = [
    ("LRP vs DeepLIFT", mc_attrs_106, mc_dl_106),
    ("LRP vs IG", mc_attrs_106, mc_ig_106),
    ("LRP vs SHAP", mc_attrs_106, mc_shap_106),
    ("DeepLIFT vs IG", mc_dl_106, mc_ig_106),
    ("DeepLIFT vs SHAP", mc_dl_106, mc_shap_106),
    ("SHAP vs IG", mc_shap_106, mc_ig_106)
]

class_names = ['left', 'right', 'both_fists', 'both_feet']

for pair_name, dict1, dict2 in mc_method_pairs:
    print(f"\n{pair_name}")
    print("-" * 50)
    agreement = compute_method_agreement(
        dict1, dict2,
        valid_subjects, criterion='kappa', n_classes=4
    )
    for cls, name in enumerate(class_names):
        correlations = [agreement[s][cls] for s in valid_subjects
                       if agreement.get(s, {}).get(cls) is not None]
        print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
              f"std={np.std(correlations):.4f}")

DeepLIFT vs IG agreement, binary, 106 subjects
--------------------------------------------------
Class left: mean_r=0.9295, std=0.0318
Class right: mean_r=0.9357, std=0.0268

DeepLIFT vs SHAP agreement, binary, 106 subjects
--------------------------------------------------
Class left: mean_r=0.2998, std=0.1541
Class right: mean_r=0.2908, std=0.1452

SHAP vs IG agreement, binary, 106 subjects
--------------------------------------------------
Class left: mean_r=0.3730, std=0.1529
Class right: mean_r=0.3567, std=0.1435

MULTICLASS — all six pairs, all four classes

LRP vs DeepLIFT
--------------------------------------------------
Class left: mean_r=0.9246, std=0.0362
Class right: mean_r=0.9376, std=0.0304
Class both_fists: mean_r=0.9360, std=0.0263
Class both_feet: mean_r=0.9348, std=0.0341

LRP vs IG
--------------------------------------------------
Class left: mean_r=0.9421, std=0.0385
Class right: mean_r=0.9536, std=0.0308
Class both_fists: mean_r=0.9582, std=0.0282
Class both_fee

### Per class group A

In [122]:
import json

base_path = ".."

with open(f"{base_path}/results/per_class_accuracy/multiclass_16ch_per_class_accuracy.json", "r") as f:
    per_class_data = json.load(f)

first_key = list(per_class_data.keys())[0]
print("Top level keys sample:", list(per_class_data.keys())[:5])
print("First entry:", first_key)
print("First entry value:", per_class_data[first_key])

Top level keys sample: ['S001', 'S002', 'S003', 'S004', 'S005']
First entry: S001
First entry value: {'kappa': {'kappa': 0.5400881057268723, 'accuracy': 0.6551724137931034, 'per_class_accuracy': {'0': 0.7272727272727273, '1': 0.4090909090909091, '2': 0.6190476190476191, '3': 0.8636363636363636}, 'confusion_matrix': [[16, 2, 4, 0], [5, 9, 3, 5], [6, 2, 13, 0], [3, 0, 0, 19]]}, 'composite': {'kappa': 0.5099454321422285, 'accuracy': 0.632183908045977, 'per_class_accuracy': {'0': 0.4090909090909091, '1': 0.5909090909090909, '2': 0.6666666666666666, '3': 0.8636363636363636}, 'confusion_matrix': [[9, 4, 9, 0], [4, 13, 2, 3], [4, 3, 14, 0], [1, 1, 1, 19]]}}


In [127]:
# Per class accuracy for all Group A subjects
# Shows which class collapsed for each subject

group_a_subjects = sorted([s for s, g in category_groups_106.items()
                           if g == 'group_a_collapse'])

print("Group A subjects — per class accuracy and collapsed class")
print("=" * 70)
print(f"{'Subject':<10} {'Left':>8} {'Right':>8} {'Fists':>8} {'Feet':>8} {'Collapsed':>12}")
print("-" * 70)

for subj in group_a_subjects:
    pca = all_per_class[subj]
    left = pca[0]
    right = pca[1]
    fists = pca[2]
    feet = pca[3]

    # Collapsed class is the one with lowest accuracy
    collapsed_idx = min(pca, key=pca.get)
    class_names = {0: 'left', 1: 'right', 2: 'both_fists', 3: 'both_feet'}
    collapsed = class_names[collapsed_idx]

    print(f"{subj:<10} {left:>8.3f} {right:>8.3f} {fists:>8.3f} "
          f"{feet:>8.3f} {collapsed:>12}")

Group A subjects — per class accuracy and collapsed class
Subject        Left    Right    Fists     Feet    Collapsed
----------------------------------------------------------------------
S006          0.250    0.429    0.591    0.174    both_feet
S014          0.500    0.652    0.167    0.476   both_fists
S016          0.682    0.273    0.083    0.333   both_fists
S017          0.522    0.318    0.083    0.476   both_fists
S021          0.500    0.571    0.125    0.846   both_fists
S024          0.636    0.304    0.045    0.609   both_fists
S037          0.739    0.182    0.273    0.286        right
S038          0.696    0.273    0.391    0.182    both_feet
S047          0.167    0.400    0.190    0.591         left
S067          0.364    0.609    0.043    0.273   both_fists
S081          0.909    0.043    0.409    0.261        right
S086          0.273    0.696    0.143    0.833   both_fists
S099          0.048    0.435    0.619    0.250         left


In [ ]:
import pandas as pd

rows = []
class_names = {0: 'left', 1: 'right', 2: 'both_fists', 3: 'both_feet'}

for subj in group_a_subjects:
    pca = all_per_class[subj]
    collapsed_idx = min(pca, key=pca.get)
    rows.append({
        'subject': subj,
        'left': round(pca[0], 3),
        'right': round(pca[1], 3),
        'both_fists': round(pca[2], 3),
        'both_feet': round(pca[3], 3),
        'collapsed_class': class_names[collapsed_idx],
        'binary_kappa': round(all_kappas[subj], 4)
    })

df_group_a = pd.DataFrame(rows)

export_path = results_path / 'group_a_per_class_accuracy.csv'
df_group_a.to_csv(export_path, index=False)
print(f"Saved to {export_path}")
print()
print(df_group_a.to_string(index=False))